# Session 13 · Data Analysis with Spark: Clustering and Forecasting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-13-spark-clustering-and-forecasting.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Data preparation: customer features

In [2]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("prep")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# 3 hidden customer types: bargain hunters, regulars, premium buyers
seg = F.when(F.col("customer_id") % 10 < 5, 0).when(F.col("customer_id") % 10 < 8, 1).otherwise(2)
orders = (spark.range(60000)
    .withColumn("customer_id", (F.col("id") % 1500).cast("int"))
    .withColumn("seg", seg)
    .withColumn("amount", F.round(F.when(F.col("seg") == 0, 800).when(F.col("seg") == 1, 2500).otherwise(9000)
                                  * F.exp(F.randn(seed=4) * 0.3), 0))
    .filter((F.col("seg") != 0) | (F.rand(seed=5) < 0.6))        # bargain hunters order less often
    .withColumn("amount", F.when(F.rand(seed=6) < 0.01, None).otherwise(F.col("amount"))))  # 1% missing

print("rows:", orders.count(), "| null amounts:", orders.filter(F.col("amount").isNull()).count())
clean = orders.na.drop(subset=["amount"]).dropDuplicates(["id"])
cust = (clean.groupBy("customer_id").agg(
            F.count("*").alias("orders"),
            F.round(F.avg("amount"), 0).alias("avg_order"),
            F.round(F.sum("amount"), 0).alias("total_spend")))
cust.summary("count", "mean", "min", "max").show()
cust.orderBy("customer_id").show(5)

rows: 48048 | null amounts: 465


+-------+-----------+------+-----------------+-----------+
|summary|customer_id|orders|        avg_order|total_spend|
+-------+-----------+------+-----------------+-----------+
|  count|       1500|  1500|             1500|       1500|
|   mean|      749.5|31.722|3087.730666666667| 115714.156|
|    min|          0|    13|            673.0|    11114.0|
|    max|       1499|    40|          10903.0|   436111.0|
+-------+-----------+------+-----------------+-----------+



+-----------+------+---------+-----------+
|customer_id|orders|avg_order|total_spend|
+-----------+------+---------+-----------+
|          0|    24|    822.0|    19723.0|
|          1|    25|    872.0|    21799.0|
|          2|    19|    869.0|    16508.0|
|          3|    22|    813.0|    17879.0|
|          4|    29|    842.0|    24426.0|
+-----------+------+---------+-----------+
only showing top 5 rows


### Example 2 · K-means segmentation with a Pipeline and silhouette

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
spark = (SparkSession.builder.master("local[2]").appName("kmeans")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

seg = F.when(F.col("customer_id") % 10 < 5, 0).when(F.col("customer_id") % 10 < 8, 1).otherwise(2)
orders = (spark.range(60000)
    .withColumn("customer_id", (F.col("id") % 1500).cast("int")).withColumn("seg", seg)
    .withColumn("amount", F.round(F.when(F.col("seg") == 0, 800).when(F.col("seg") == 1, 2500).otherwise(9000)
                                  * F.exp(F.randn(seed=4) * 0.3), 0))
    .filter((F.col("seg") != 0) | (F.rand(seed=5) < 0.6)))
cust = orders.groupBy("customer_id").agg(F.count("*").alias("orders"),
                                         F.avg("amount").alias("avg_order")).cache()

assembler = VectorAssembler(inputCols=["orders", "avg_order"], outputCol="raw")
scaler = StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True)
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette")

print(" k  silhouette")
for k in [2, 3, 4, 5, 6]:
    model = Pipeline(stages=[assembler, scaler, KMeans(k=k, seed=42, featuresCol="features")]).fit(cust)
    print(f"{k:>2}  {evaluator.evaluate(model.transform(cust)):.3f}")

best = Pipeline(stages=[assembler, scaler, KMeans(k=3, seed=42, featuresCol="features")]).fit(cust)
profiles = (best.transform(cust).groupBy("prediction")
            .agg(F.count("*").alias("customers"), F.round(F.avg("orders"), 1).alias("avg_orders"),
                 F.round(F.avg("avg_order"), 0).alias("avg_order_value"))
            .orderBy("avg_order_value"))
profiles.show()

 k  silhouette


 2  0.741


 3  0.960


 4  0.849


 5  0.792


 6  0.797


+----------+---------+----------+---------------+
|prediction|customers|avg_orders|avg_order_value|
+----------+---------+----------+---------------+
|         0|      749|      24.1|          838.0|
|         1|      451|      40.0|         2613.0|
|         2|      300|      40.0|         9415.0|
+----------+---------+----------+---------------+



### Example 3 · Forecasting daily sales with lag features

In [4]:
import math
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
spark = (SparkSession.builder.master("local[2]").appName("forecast")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# 2 years of daily sales: trend + weekly pattern (Friday and weekend peaks) + noise
days = (spark.range(730).withColumnRenamed("id", "t")
        .withColumn("date", F.date_add(F.lit("2024-11-01").cast("date"), F.col("t").cast("int")))
        .withColumn("dow", F.dayofweek("date"))
        .withColumn("sales", F.round(200 + 0.15 * F.col("t")
                     + F.when(F.col("dow").isin(6, 7, 1), 60).otherwise(0)
                     + 20 * F.sin(2 * math.pi * F.col("t") / 365) + F.randn(seed=7) * 12, 1)))

w = Window.orderBy("t")
feat = (days.withColumn("lag_1", F.lag("sales", 1).over(w))
            .withColumn("lag_7", F.lag("sales", 7).over(w))
            .withColumn("roll_7", F.avg("sales").over(w.rowsBetween(-7, -1)))
            .withColumn("weekend", F.col("dow").isin(6, 7, 1).cast("int"))
            .na.drop())
cut = "2026-08-01"
train, test = feat.filter(F.col("date") < cut), feat.filter(F.col("date") >= cut)
print(f"train days: {train.count()}, test days: {test.count()} (test starts {cut})")

va = VectorAssembler(inputCols=["t", "lag_1", "lag_7", "roll_7", "weekend"], outputCol="features")
tr, te = va.transform(train), va.transform(test)
models = {"Linear regression": LinearRegression(labelCol="sales"),
          "Random forest": RandomForestRegressor(labelCol="sales", numTrees=50, seed=1)}

def metrics(df, pred):
    r = {m: RegressionEvaluator(labelCol="sales", predictionCol=pred, metricName=m).evaluate(df) for m in ["mae", "rmse"]}
    r["mape"] = df.select(F.avg(F.abs((F.col("sales") - F.col(pred)) / F.col("sales"))) * 100).first()[0]
    return r

print(f"\n{'model':<26}{'MAE':>7}{'RMSE':>7}{'MAPE %':>8}")
base = te.withColumn("naive", F.col("lag_7"))
r = metrics(base, "naive"); print(f"{'Naive (same day last wk)':<26}{r['mae']:>7.1f}{r['rmse']:>7.1f}{r['mape']:>8.1f}")
for name, m in models.items():
    p = m.fit(tr).transform(te)
    r = metrics(p, "prediction"); print(f"{name:<26}{r['mae']:>7.1f}{r['rmse']:>7.1f}{r['mape']:>8.1f}")

lr = models["Linear regression"].fit(tr)
print("\nlast 5 test days:")
lr.transform(te).select("date", "sales", F.round("prediction", 1).alias("forecast")).orderBy(F.desc("date")).limit(5).orderBy("date").show()

train days: 631, test days: 92 (test starts 2026-08-01)

model                         MAE   RMSE  MAPE %


Naive (same day last wk)     13.5   17.2     4.3


Linear regression            10.2   12.6     3.2


Random forest                14.1   17.5     4.3



last 5 test days:


+----------+-----+--------+
|      date|sales|forecast|
+----------+-----+--------+
|2026-10-27|315.1|   305.7|
|2026-10-28|319.7|   306.8|
|2026-10-29|309.6|   308.7|
|2026-10-30|376.9|   368.8|
|2026-10-31|350.7|   373.6|
+----------+-----+--------+

